In [1]:
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path("/Users/jonathantran/Desktop/nba_economics")
STAGING_DIR = PROJECT_ROOT / "data" / "staging"
CANONICAL_DIR = PROJECT_ROOT / "data" / "canonical"

s01 = pd.read_csv(
    STAGING_DIR / "standings" / "s01_standings.csv"
)

s08 = pd.read_csv(
    STAGING_DIR / "team_payroll" / "s08_team_payroll.csv"
)

s02 = pd.read_csv(STAGING_DIR / "advanced_stats" / "s02_advanced_stats.csv")

s03 = pd.read_csv(STAGING_DIR / "postseason" / "s03_team_season_postseason.csv")



In [4]:
sources = {
    "S01": (s01, "season", "team"),
    "S02": (s02, "season", "Team"),
    "S03": (s03, "season", "team"),
    "S08": (s08, "Season", "Team_Name")
}

# Identify the exact column names in S02 and S03.
print("S02:", s02.columns.tolist())
print("S03:", s03.columns.tolist())

S02: ['Rank', 'Team', 'Age', 'W', 'L', 'PW', 'PL', 'MOV', 'SOS', 'SRS', 'ORtg', 'DRtg', 'NRtg', 'Pace', 'FTr', '3PAr', 'TS%', 'off_eFG%', 'off_TOV%', 'off_ORB%', 'off_FT/FGA', 'def_eFG%', 'def_TOV%', 'def_DRB%', 'def_FT/FGA', 'Arena', 'Attend.', 'Attend./G', 'season', 'year']
S03: ['season', 'year', 'team', 'seed', 'playoff_series_won', 'round_score', 'furthest_round']


In [5]:
for source_name, (df, season_col, team_col) in sources.items():

    print(f"\n{'=' * 45}")
    print(source_name)
    print("=" * 45)

    print("Total rows:", len(df))

    print(
        "Missing team identifiers:",
        df[team_col].isna().sum()
    )

    print(
        "Missing seasons:",
        df[season_col].isna().sum()
    )

    print(
        "Unique team-season combinations:",
        df[[team_col, season_col]]
        .drop_duplicates()
        .shape[0]
    )

    print(
        "Duplicate team-season keys:",
        df.duplicated([team_col, season_col]).sum()
    )

    print(
        "Season range:",
        sorted(df[season_col].dropna().astype(str).unique())[0],
        "to",
        sorted(df[season_col].dropna().astype(str).unique())[-1]
    )

    print(
        "Unique seasons:",
        df[season_col].nunique()
    )

    print(
        "Season examples:",
        df[season_col].dropna().astype(str).unique()[:5]
    )

    duplicates = df[
        df.duplicated([team_col, season_col], keep=False)
    ]

    if not duplicates.empty:
        print("\nSample duplicate records:")
        display(duplicates.head(10))


S01
Total rows: 776
Missing team identifiers: 0
Missing seasons: 0
Unique team-season combinations: 776
Duplicate team-season keys: 0
Season range: 2000-2001 to 2025-2026
Unique seasons: 26
Season examples: <StringArray>
['2000-2001', '2001-2002', '2002-2003', '2003-2004', '2004-2005']
Length: 5, dtype: str

S02
Total rows: 776
Missing team identifiers: 0
Missing seasons: 0
Unique team-season combinations: 776
Duplicate team-season keys: 0
Season range: 2000-2001 to 2025-2026
Unique seasons: 26
Season examples: <StringArray>
['2000-2001', '2001-2002', '2002-2003', '2003-2004', '2004-2005']
Length: 5, dtype: str

S03
Total rows: 416
Missing team identifiers: 0
Missing seasons: 0
Unique team-season combinations: 416
Duplicate team-season keys: 0
Season range: 2000-2001 to 2025-2026
Unique seasons: 26
Season examples: <StringArray>
['2000-2001', '2001-2002', '2002-2003', '2003-2004', '2004-2005']
Length: 5, dtype: str

S08
Total rows: 776
Missing team identifiers: 0
Missing seasons: 0
Un

In [6]:
aliases = pd.read_csv(
    CANONICAL_DIR / "franchise_season_aliases.csv"
)

print("Shape:", aliases.shape)
print("Columns:", aliases.columns.tolist())

display(aliases.head())

Shape: (776, 5)
Columns: ['franchise_id', 'season', 'team_abbreviation', 'full_team_name', 'short_team_name']


,franchise_id,season,team_abbreviation,full_team_name,short_team_name
0,1610612737,2000-2001,ATL,Atlanta Hawks,Hawks
1,1610612738,2000-2001,BOS,Boston Celtics,Celtics
2,1610612739,2000-2001,CLE,Cleveland Cavaliers,Cavaliers
3,1610612741,2000-2001,CHI,Chicago Bulls,Bulls
4,1610612742,2000-2001,DAL,Dallas Mavericks,Mavericks


In [7]:
assert len(aliases) == 776

assert not aliases.duplicated(
    ["franchise_id", "season"]
).any()

assert aliases[
    ["franchise_id", "season", "full_team_name", "short_team_name"]
].notna().any(axis=1).all()

print("C02 alias table loaded and validated.")

C02 alias table loaded and validated.


In [8]:
# Prepare source-specific keys without changing staging data.
source_keys = {
    "S01": (
        s01.assign(
            team_clean=s01["team"].str.replace(
                r"\*$", "", regex=True
            ).str.strip()
        ),
        "season",
        "team_clean",
        "full_team_name"
    ),
    "S02": (
        s02.assign(
            team_clean=s02["Team"].str.replace(
                r"\*$", "", regex=True
            ).str.strip()
        ),
        "season",
        "team_clean",
        "full_team_name"
    ),
    "S03": (
        s03.assign(
            team_clean=s03["team"].str.replace(
                r"\*$", "", regex=True
            ).str.strip()
        ),
        "season",
        "team_clean",
        "full_team_name"
    ),
    "S08": (
        s08,
        "Season",
        "Team_Name",
        "short_team_name"
    )
}

for name, (df, season_col, team_col, alias_col) in source_keys.items():

    test = df.merge(
        aliases[["franchise_id", "season", alias_col]],
        left_on=[season_col, team_col],
        right_on=["season", alias_col],
        how="left",
        validate="one_to_one",
        indicator=True
    )

    print(
        name,
        "| Matched:", (test["_merge"] == "both").sum(),
        "| Unmatched:", (test["_merge"] == "left_only").sum()
    )

    if (test["_merge"] == "left_only").any():
        display(
            test.loc[
                test["_merge"] == "left_only",
                [season_col, team_col]
            ].head(15)
        )

S01 | Matched: 776 | Unmatched: 0
S02 | Matched: 776 | Unmatched: 0
S03 | Matched: 416 | Unmatched: 0
S08 | Matched: 776 | Unmatched: 0


In [9]:
canonical_sources = {}

for name, (df, season_col, team_col, alias_col) in source_keys.items():

    mapped = df.merge(
        aliases[["franchise_id", "season", alias_col]],
        left_on=[season_col, team_col],
        right_on=["season", alias_col],
        how="left",
        validate="one_to_one"
    )

    # Standardize season column for S08.
    if season_col != "season":
        mapped = mapped.drop(columns=["season"])
        mapped = mapped.rename(columns={season_col: "season"})

    # Remove the historical name used solely for matching.
    mapped = mapped.drop(columns=[team_col, alias_col])

    # Remove year, since season is our canonical time key.
    mapped = mapped.drop(columns=["year", "Year"], errors="ignore")

    canonical_sources[name] = mapped

    print(name, mapped.shape)

S01 (776, 11)
S02 (776, 30)
S03 (416, 7)
S08 (776, 5)


In [10]:
team_seasons = canonical_sources["S01"].copy()

for name in ["S02", "S03", "S08"]:

    source = canonical_sources[name]

    team_seasons = team_seasons.merge(
        source,
        on=["franchise_id", "season"],
        how="left",
        validate="one_to_one",
        suffixes=("", f"_{name.lower()}")
    )

    print(name, "merged:", team_seasons.shape)

S02 merged: (776, 39)
S03 merged: (776, 44)
S08 merged: (776, 47)


In [11]:
print("Shape:", team_seasons.shape)

print(
    "Duplicate franchise-season keys:",
    team_seasons.duplicated(
        ["franchise_id", "season"]
    ).sum()
)

print(
    "Missing franchise IDs:",
    team_seasons["franchise_id"].isna().sum()
)

print("\nColumns:")
print(team_seasons.columns.tolist())

display(team_seasons.head())

Shape: (776, 47)
Duplicate franchise-season keys: 0
Missing franchise IDs: 0

Columns:
['team', 'W', 'L', 'W/L%', 'GB', 'PS/G', 'PA/G', 'SRS', 'conference', 'season', 'franchise_id', 'Rank', 'Team', 'Age', 'W_s02', 'L_s02', 'PW', 'PL', 'MOV', 'SOS', 'SRS_s02', 'ORtg', 'DRtg', 'NRtg', 'Pace', 'FTr', '3PAr', 'TS%', 'off_eFG%', 'off_TOV%', 'off_ORB%', 'off_FT/FGA', 'def_eFG%', 'def_TOV%', 'def_DRB%', 'def_FT/FGA', 'Arena', 'Attend.', 'Attend./G', 'team_s03', 'seed', 'playoff_series_won', 'round_score', 'furthest_round', 'Team_Payroll', 'Salary_Cap_Hit', 'Salary_Cap_Space']


,team,W,L,W/L%,GB,PS/G,PA/G,SRS,conference,season,...,Attend.,Attend./G,team_s03,seed,playoff_series_won,round_score,furthest_round,Team_Payroll,Salary_Cap_Hit,Salary_Cap_Space
0,Philadelphia 76ers*,56,26,0.683,0.0,94.7,90.4,3.64,East,2000-2001,...,805692.0,19651.0,Philadelphia 76ers,1.0,3.0,4.0,NBA Finals,51085458,51085458,NaN
1,Miami Heat*,50,32,0.610,6.0,88.9,86.6,1.73,East,2000-2001,...,678186.0,16541.0,Miami Heat,3.0,0.0,1.0,First Round,73472329,73472329,NaN
2,New York Knicks*,48,34,0.585,8.0,88.7,86.1,1.98,East,2000-2001,...,810283.0,19763.0,New York Knicks,4.0,0.0,1.0,First Round,74007738,74007738,NaN
3,Orlando Magic*,43,39,0.524,13.0,97.5,96.5,0.39,East,2000-2001,...,605031.0,14757.0,Orlando Magic,7.0,0.0,1.0,First Round,37459860,37459860,NaN
4,Boston Celtics,36,46,0.439,20.0,94.6,96.8,-2.40,East,2000-2001,...,629201.0,15346.0,NaN,NaN,NaN,NaN,NaN,51478707,51478707,NaN


In [12]:
assert len(team_seasons) == 776

assert not team_seasons.duplicated(
    ["franchise_id", "season"]
).any()

assert team_seasons[
    ["franchise_id", "season"]
].notna().all().all()

print("C04 structural validation passed.")

C04 structural validation passed.


In [13]:
team_seasons["team"] = (
    team_seasons["team"]
    .str.replace(r"\*$", "", regex=True)
    .str.strip()
)

In [14]:
team_seasons = team_seasons.drop(
    columns=["Team", "team_s03"]
)

In [15]:
for col1, col2 in [
    ("W", "W_s02"),
    ("L", "L_s02"),
    ("SRS", "SRS_s02")
]:
    matches = (
        team_seasons[col1].eq(team_seasons[col2])
        | (
            team_seasons[col1].isna()
            & team_seasons[col2].isna()
        )
    )

    print(
        f"{col1} vs {col2}:",
        (~matches).sum(),
        "mismatches"
    )

W vs W_s02: 0 mismatches
L vs L_s02: 0 mismatches
SRS vs SRS_s02: 0 mismatches


In [16]:
team_seasons = team_seasons.drop(
    columns=["W_s02", "L_s02", "SRS_s02"]
)

In [17]:
playoff_indicator = (
    s01["team"].str.endswith("*").fillna(False)
)

print(
    "S01 playoff teams:",
    playoff_indicator.sum()
)

print(
    "C04 playoff records:",
    team_seasons["furthest_round"].notna().sum()
)

S01 playoff teams: 416
C04 playoff records: 416


In [18]:
team_seasons["made_playoffs"] = (
    team_seasons["furthest_round"].notna()
)

playoff_check = s01[
    ["season", "team"]
].copy()

playoff_check["made_playoffs_s01"] = (
    playoff_check["team"].str.endswith("*").fillna(False)
)

playoff_check["team"] = (
    playoff_check["team"]
    .str.replace(r"\*$", "", regex=True)
    .str.strip()
)

playoff_check = playoff_check.merge(
    team_seasons[
        ["season", "team", "made_playoffs"]
    ],
    on=["season", "team"],
    validate="one_to_one"
)

print(
    "Playoff indicator disagreements:",
    (
        playoff_check["made_playoffs_s01"]
        != playoff_check["made_playoffs"]
    ).sum()
)

Playoff indicator disagreements: 0


In [19]:
team_seasons["playoff_series_won"] = (
    team_seasons["playoff_series_won"].fillna(0).astype(int)
)

In [20]:
financial_cols = [
    "Team_Payroll",
    "Salary_Cap_Hit",
    "Salary_Cap_Space"
]

print(team_seasons[financial_cols].isna().sum())

print(
    "Payroll vs cap-hit mismatches:",
    (
        team_seasons["Team_Payroll"]
        != team_seasons["Salary_Cap_Hit"]
    ).sum()
)

Team_Payroll          0
Salary_Cap_Hit        0
Salary_Cap_Space    686
dtype: int64
Payroll vs cap-hit mismatches: 0


In [21]:
team_seasons = team_seasons.rename(columns={
    "team": "team_name",
    "W": "wins",
    "L": "losses",
    "W/L%": "win_pct",
    "GB": "games_behind",
    "PS/G": "points_per_game",
    "PA/G": "opponent_points_per_game",
    "SRS": "simple_rating_system",
    "Rank": "league_rank",
    "Age": "average_age",
    "PW": "pythagorean_wins",
    "PL": "pythagorean_losses",
    "MOV": "margin_of_victory",
    "SOS": "strength_of_schedule",
    "ORtg": "offensive_rating",
    "DRtg": "defensive_rating",
    "NRtg": "net_rating",
    "Pace": "pace",
    "FTr": "free_throw_rate",
    "3PAr": "three_point_attempt_rate",
    "TS%": "true_shooting_pct",
    "Arena": "arena",
    "Attend.": "attendance",
    "Attend./G": "attendance_per_game",
    "Team_Payroll": "team_payroll",
    "Salary_Cap_Hit": "salary_cap_hit",
    "Salary_Cap_Space": "salary_cap_space"
})

In [22]:
for col1, col2 in [
    ("W", "W_s02"),
    ("L", "L_s02"),
    ("SRS", "SRS_s02")
]:
    mismatches = (
        team_seasons[col1].ne(team_seasons[col2])
        & ~(
            team_seasons[col1].isna()
            & team_seasons[col2].isna()
        )
    ).sum()

    print(f"{col1} vs {col2}: {mismatches} mismatches")

KeyError: 'W'

In [23]:
print(team_seasons.columns.tolist())

['team_name', 'wins', 'losses', 'win_pct', 'games_behind', 'points_per_game', 'opponent_points_per_game', 'simple_rating_system', 'conference', 'season', 'franchise_id', 'league_rank', 'average_age', 'pythagorean_wins', 'pythagorean_losses', 'margin_of_victory', 'strength_of_schedule', 'offensive_rating', 'defensive_rating', 'net_rating', 'pace', 'free_throw_rate', 'three_point_attempt_rate', 'true_shooting_pct', 'off_eFG%', 'off_TOV%', 'off_ORB%', 'off_FT/FGA', 'def_eFG%', 'def_TOV%', 'def_DRB%', 'def_FT/FGA', 'arena', 'attendance', 'attendance_per_game', 'seed', 'playoff_series_won', 'round_score', 'furthest_round', 'team_payroll', 'salary_cap_hit', 'salary_cap_space', 'made_playoffs']


In [24]:
# Identify whether the columns have already been renamed.
column_pairs = [
    ("wins" if "wins" in team_seasons.columns else "W", "W_s02"),
    ("losses" if "losses" in team_seasons.columns else "L", "L_s02"),
    (
        "simple_rating_system"
        if "simple_rating_system" in team_seasons.columns
        else "SRS",
        "SRS_s02"
    )
]

for col1, col2 in column_pairs:

    if col2 not in team_seasons.columns:
        print(f"{col2} is missing — comparison may already have been cleaned up.")
        continue

    mismatches = (
        team_seasons[col1].ne(team_seasons[col2])
        & ~(
            team_seasons[col1].isna()
            & team_seasons[col2].isna()
        )
    ).sum()

    print(f"{col1} vs {col2}: {mismatches} mismatches")

W_s02 is missing — comparison may already have been cleaned up.
L_s02 is missing — comparison may already have been cleaned up.
SRS_s02 is missing — comparison may already have been cleaned up.


In [25]:
# Create temporary copies for validation.
check_s01 = s01.copy()
check_s02 = s02.copy()

# Standardize team names for matching.
check_s01["team_clean"] = (
    check_s01["team"]
    .str.replace(r"\*$", "", regex=True)
    .str.strip()
)

check_s02["team_clean"] = (
    check_s02["Team"]
    .str.replace(r"\*$", "", regex=True)
    .str.strip()
)

# Join only the columns we need to compare.
comparison = check_s01[
    ["season", "team_clean", "W", "L", "SRS"]
].merge(
    check_s02[
        ["season", "team_clean", "W", "L", "SRS"]
    ],
    on=["season", "team_clean"],
    how="outer",
    suffixes=("_s01", "_s02"),
    validate="one_to_one",
    indicator=True
)

print("Join coverage:")
print(comparison["_merge"].value_counts())

for stat in ["W", "L", "SRS"]:
    mismatches = (
        comparison[f"{stat}_s01"].ne(comparison[f"{stat}_s02"])
        & ~(
            comparison[f"{stat}_s01"].isna()
            & comparison[f"{stat}_s02"].isna()
        )
    )

    print(f"{stat}: {mismatches.sum()} mismatches")

    if mismatches.any():
        display(
            comparison.loc[
                mismatches,
                ["season", "team_clean",
                 f"{stat}_s01", f"{stat}_s02"]
            ].head(10)
        )

Join coverage:
_merge
both          776
left_only       0
right_only      0
Name: count, dtype: int64
W: 0 mismatches
L: 0 mismatches
SRS: 0 mismatches


In [26]:
# Confirm the expected structure.
assert len(team_seasons) == 776

assert team_seasons["season"].nunique() == 26

assert team_seasons["franchise_id"].nunique() == 30

assert not team_seasons.duplicated(
    ["franchise_id", "season"]
).any()

assert team_seasons[
    ["franchise_id", "season", "team_name"]
].notna().all().all()

assert team_seasons["made_playoffs"].sum() == 416

assert team_seasons["team_payroll"].notna().all()

# Confirm the playoff asterisks have been removed.
assert not team_seasons["team_name"].str.contains(
    r"\*$", regex=True
).any()

# Confirm no leftover duplicate source columns.
assert not any(
    col in team_seasons.columns
    for col in ["W_s02", "L_s02", "SRS_s02", "Team", "team_s03"]
)

print("C04 FINAL VALIDATION PASSED")
print("Shape:", team_seasons.shape)

C04 FINAL VALIDATION PASSED
Shape: (776, 43)


In [27]:
output_path = CANONICAL_DIR / "team_seasons.csv"

team_seasons.to_csv(output_path, index=False)

print("Exported:", output_path)

Exported: /Users/jonathantran/Desktop/nba_economics/data/canonical/team_seasons.csv
